# Riemann data pipeline — consolidated 00–05

Single Colab pipeline: 00 environment → integrity → 01 acquire → 02 describe → 03 unfold → 04 surrogates → 05 compare.

Artifact rule: existing raw/derived artifacts are never overwritten. Existing artifacts are verified against data/manifest.json. A missing derived artifact is created once, then hashed and recorded in the local manifest; any later mismatch is a hard error.

## 00 — Environment / Colab restart

Run all once. If the package is not installed from this checkout, this cell installs it and terminates the runtime. Run all again after Colab reconnects. The repository cd is performed on every run, including the post-restart run.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPO_BASE = Path('/content/nicht-riemann-data').resolve()
REPO_URL = 'https://github.com/nicht-organization/nicht-riemann-data.git'

if not REPO_BASE.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_BASE)], check=True)
elif not (REPO_BASE / '.git').is_dir():
    raise RuntimeError(f'Path exists but is not a Git repository: {REPO_BASE}')

os.chdir(REPO_BASE)
print('repo pwd:', Path.cwd())
subprocess.run(['git', 'status', '--short'], check=True)
subprocess.run(['git', 'branch', '--show-current'], check=True)

package_spec = importlib.util.find_spec('nicht_riemann_data')
package_from_checkout = False
if package_spec is not None and package_spec.origin is not None:
    try:
        package_from_checkout = Path(package_spec.origin).resolve().is_relative_to(REPO_BASE / 'src')
    except ValueError:
        package_from_checkout = False

if not package_from_checkout:
    print('Installing editable package from checkout; restarting runtime...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
    print('pwd immediately before os._exit(0):')
    subprocess.run(['pwd'], check=True)
    os._exit(0)

print('pwd after restart/import path setup:')
subprocess.run(['pwd'], check=True)
from nicht_riemann_data.transforms import spacings
from nicht_riemann_data.diagnostics import describe
print('Package import: OK')

## Integrity helpers


In [ ]:
import hashlib
import json
import numpy as np

MANIFEST_FILE = REPO_BASE / 'data/manifest.json'
DATA_DIR = REPO_BASE / 'data/raw'
DERIVED_DIR = REPO_BASE / 'data/derived'
DATA_DIR.mkdir(parents=True, exist_ok=True)
DERIVED_DIR.mkdir(parents=True, exist_ok=True)
assert MANIFEST_FILE.is_file(), f'Missing manifest: {MANIFEST_FILE}'

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def load_manifest() -> dict:
    with MANIFEST_FILE.open('r', encoding='utf-8') as f:
        value = json.load(f)
    assert isinstance(value, dict), 'manifest root must be an object'
    assert isinstance(value.get('datasets'), dict), 'manifest.datasets must be an object'
    assert isinstance(value.get('derived'), dict), 'manifest.derived must be an object'
    return value

def save_manifest(manifest: dict) -> None:
    tmp = MANIFEST_FILE.with_suffix('.json.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(manifest, f, indent=2)
        f.write('\n')
    tmp.replace(MANIFEST_FILE)

manifest = load_manifest()
raw_spec = manifest['datasets']['odlyzko_zeros1']
derived_spec = manifest['derived']['unfolded_spacings']
assert raw_spec['local_file'] == 'data/raw/zeros1'
assert derived_spec['local_file'] == 'data/derived/unfolded_spacings.float64'
print('manifest loaded: OK')

## 01 — Acquire

The raw artifact is downloaded only when absent. Whether downloaded or pre-existing, it is then checked against the manifest byte count and SHA-256.

In [ ]:
import urllib.request

RAW_FILE = DATA_DIR / 'zeros1'
if RAW_FILE.exists():
    print(f'Existing raw artifact: {RAW_FILE}')
else:
    print(f'Downloading raw artifact: {raw_spec["url"]}')
    urllib.request.urlretrieve(raw_spec['url'], RAW_FILE)

actual_bytes = RAW_FILE.stat().st_size
actual_sha256 = sha256(RAW_FILE)
assert actual_bytes == raw_spec['raw_bytes'], f'Raw artifact size mismatch: {actual_bytes} != {raw_spec["raw_bytes"]}'
assert actual_sha256 == raw_spec['sha256'], f'Raw artifact SHA-256 mismatch: {actual_sha256} != {raw_spec["sha256"]}'
print('Verified raw artifact:', RAW_FILE)
print('bytes:', actual_bytes)
print('SHA-256:', actual_sha256)

In [ ]:
gamma = np.loadtxt(RAW_FILE, dtype=np.float64)
assert gamma.ndim == 1
assert gamma.dtype == np.float64
assert len(gamma) == raw_spec['records']
assert np.all(np.isfinite(gamma))
assert np.all(np.diff(gamma) > 0)
assert np.isclose(gamma[0], raw_spec['first_value'])
assert np.isclose(gamma[-1], raw_spec['last_value'])
print('N:', len(gamma))
print('first:', gamma[:5])
print('last:', gamma[-5:])
print('raw data validation: OK')

## 02 — Describe


In [ ]:
delta = spacings(gamma)
assert delta.shape == (len(gamma) - 1,)
assert np.all(np.isfinite(delta))
assert np.all(delta > 0)
print('gamma:')
print(describe(gamma))
print('\ndelta:')
print(describe(delta))

gamma_range = gamma[-1] - gamma[0]
mean_spacing = np.mean(delta)
range_per_spacing = gamma_range / len(delta)
print('\nrange:', gamma_range)
print('mean spacing:', mean_spacing)
print('range / number of spacings:', range_per_spacing)
assert np.isclose(mean_spacing, range_per_spacing)

In [ ]:
percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]
for p, value in zip(percentile_levels, np.percentile(delta, percentile_levels)):
    print(f'{p:>3}% : {value:.12f}')

BLOCK_SIZE = 1000
num_blocks = len(delta) // BLOCK_SIZE
block_means = np.array([np.mean(delta[i * BLOCK_SIZE:(i + 1) * BLOCK_SIZE]) for i in range(num_blocks)])
remainder_delta = delta[num_blocks * BLOCK_SIZE:]
print('\nblock size:', BLOCK_SIZE)
print('full blocks:', num_blocks)
print('remainder:', len(remainder_delta))
print('first block means:', block_means[:10])
print('last block means:', block_means[-10:])
print('local mean min:', block_means.min())
print('local mean max:', block_means.max())
print('local mean ratio max/min:', block_means.max() / block_means.min())

In [ ]:
local_residuals = np.empty_like(delta)
for i in range(num_blocks):
    start = i * BLOCK_SIZE
    end = start + BLOCK_SIZE
    local_residuals[start:end] = delta[start:end] - block_means[i]
if len(remainder_delta):
    local_residuals[num_blocks * BLOCK_SIZE:] = remainder_delta - np.mean(remainder_delta)
assert local_residuals.shape == delta.shape
print('local-mean residual std:', np.std(local_residuals))
print('local-mean residual min:', np.min(local_residuals))
print('local-mean residual max:', np.max(local_residuals))

## 03 — Unfold

Use the same 1000-spacing local baseline defined in the description stage. Existing derived artifact → verify only. Missing derived artifact → create once, hash it, and record the actual bytes/records/SHA-256 in the local manifest.

In [ ]:
UNFOLDED_FILE = REPO_BASE / derived_spec['local_file']
BLOCK_SIZE = 1000
num_blocks = len(delta) // BLOCK_SIZE

if UNFOLDED_FILE.exists():
    actual_bytes = UNFOLDED_FILE.stat().st_size
    actual_sha256 = sha256(UNFOLDED_FILE)
    assert actual_bytes == derived_spec['bytes'], f'Derived artifact size mismatch: {actual_bytes} != {derived_spec["bytes"]}'
    assert actual_sha256 == derived_spec['sha256'], f'Derived artifact SHA-256 mismatch: {actual_sha256} != {derived_spec["sha256"]}'
    print('Verified existing derived artifact:', UNFOLDED_FILE)
else:
    local_mean = np.empty_like(delta)
    for i in range(num_blocks):
        start = i * BLOCK_SIZE
        end = start + BLOCK_SIZE
        local_mean[start:end] = np.mean(delta[start:end])
    remainder_start = num_blocks * BLOCK_SIZE
    if remainder_start < len(delta):
        local_mean[remainder_start:] = np.mean(delta[remainder_start:])
    unfolded = delta / local_mean
    assert unfolded.shape == delta.shape
    assert np.all(np.isfinite(unfolded))
    assert np.all(unfolded > 0)
    unfolded.astype(np.float64).tofile(UNFOLDED_FILE)
    actual_bytes = UNFOLDED_FILE.stat().st_size
    actual_sha256 = sha256(UNFOLDED_FILE)
    assert actual_bytes == derived_spec['bytes'], f'New derived artifact size mismatch: {actual_bytes} != {derived_spec["bytes"]}'
    derived_spec['sha256'] = actual_sha256
    derived_spec['bytes'] = actual_bytes
    derived_spec['records'] = len(unfolded)
    save_manifest(manifest)
    print('Created and recorded derived artifact:', UNFOLDED_FILE)

assert UNFOLDED_FILE.is_file()
unfolded = np.fromfile(UNFOLDED_FILE, dtype=np.float64)
assert len(unfolded) == derived_spec['records']
assert unfolded.shape == delta.shape
assert np.all(np.isfinite(unfolded))
assert np.all(unfolded > 0)
print('unfolded:', len(unfolded))
print('mean:', np.mean(unfolded))
print('std:', np.std(unfolded))
print('derived SHA-256:', sha256(UNFOLDED_FILE))

## 04 — Surrogates

Surrogates are in-memory comparison controls; they do not overwrite the observed derived artifact.

In [ ]:
SEED = 20260831
rng = np.random.default_rng(SEED)
surrogate_shuffle = unfolded.copy()
rng.shuffle(surrogate_shuffle)
assert np.array_equal(np.sort(surrogate_shuffle), np.sort(unfolded))
surrogate_iid = rng.choice(unfolded, size=len(unfolded), replace=True)
assert surrogate_iid.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_iid))
assert np.all(surrogate_iid > 0)
surrogate_uniform = rng.uniform(0.0, 2.0, size=len(unfolded))
assert surrogate_uniform.shape == unfolded.shape
assert np.all(np.isfinite(surrogate_uniform))
assert np.all(surrogate_uniform >= 0)
datasets = {'observed': unfolded, 'shuffle': surrogate_shuffle, 'iid': surrogate_iid, 'uniform': surrogate_uniform}
print('seed:', SEED)
for name, values in datasets.items():
    print(f'{name:>8} : mean={np.mean(values):.6f} std={np.std(values):.6f} min={np.min(values):.6f} max={np.max(values):.6f}')

In [ ]:
percentile_levels = [0, 1, 5, 25, 50, 75, 95, 99, 100]
for name, values in datasets.items():
    print(f'\n{name}')
    for p, value in zip(percentile_levels, np.percentile(values, percentile_levels)):
        print(f'{p:>3}% : {value:.12f}')

## 05 — Compare

Comparison is terminal: it consumes the verified observed artifact and the stage-04 surrogates. No acquisition, unfolding, or artifact creation occurs here.

In [ ]:
def lag1_correlation(values):
    return float(np.corrcoef(values[:-1], values[1:])[0, 1])

for name, values in datasets.items():
    assert values.ndim == 1
    assert len(values) == len(unfolded)
    assert np.all(np.isfinite(values))
    print(f'{name:>8} : lag1={lag1_correlation(values): .8f}')

assert np.array_equal(np.sort(surrogate_shuffle), np.sort(unfolded))
assert np.isclose(np.mean(surrogate_shuffle), np.mean(unfolded))
assert np.isclose(np.std(surrogate_shuffle), np.std(unfolded))
print('shuffle control: OK')

In [ ]:
print('=== PIPELINE 00–05 COMPLETE ===')
print('raw artifact verified:', RAW_FILE)
print('derived artifact verified:', UNFOLDED_FILE)
print('records:', len(unfolded))
print('manifest SHA entry:', manifest['derived']['unfolded_spacings']['sha256'])